In [9]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("test_item") \
    .config("spark.jars", "/opt/spark/jars/postgresql.jar") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark OK :", spark.version)

Spark OK : 3.5.2


In [14]:
df_raw = spark.read.format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/warehouse_db") \
    .option("dbtable", '"bronze"."Item"') \
    .option("user", "warehouse") \
    .option("password", "warehouse") \
    .option("driver", "org.postgresql.Driver") \
    .load()

print("Nombre de lignes :", df_raw.count())
df_raw.printSchema()
df_raw.show(5)

Nombre de lignes : 66873
root
 |-- _airbyte_raw_id: string (nullable = true)
 |-- _airbyte_extracted_at: timestamp (nullable = true)
 |-- _airbyte_meta: string (nullable = true)
 |-- _airbyte_generation_id: long (nullable = true)
 |-- Id: long (nullable = true)
 |-- Code: string (nullable = true)
 |-- Note: string (nullable = true)
 |-- Color: string (nullable = true)
 |-- Depth: decimal(38,18) (nullable = true)
 |-- IsKit: boolean (nullable = true)
 |-- Width: decimal(38,18) (nullable = true)
 |-- Design: string (nullable = true)
 |-- Height: decimal(38,18) (nullable = true)
 |-- Weight: decimal(38,18) (nullable = true)
 |-- OnOrder: boolean (nullable = true)
 |-- TVARate: decimal(38,18) (nullable = true)
 |-- Diameter: decimal(38,18) (nullable = true)
 |-- IdFamily: long (nullable = true)
 |-- IdNature: long (nullable = true)
 |-- TecDocId: long (nullable = true)
 |-- BarCode1D: string (nullable = true)
 |-- BarCode2D: string (nullable = true)
 |-- CostPrice: decimal(38,18) (nullable

In [3]:
from pyspark.sql import functions as F

df_filtered = df_raw.filter(
    (F.col("IsDeleted") == 0) & (F.col("IsForSales") == 1)
)
print("Après filtrage :", df_filtered.count())
df_filtered.show(5)

Après filtrage : 65678
+--------------------+---------------------+--------------------+----------------------+-----+----------+----+-----+-----+-----+-----+------+------+------+-------+-------+--------+--------+--------+---------+----------+---------+--------------------+---------+---------+---------+----------+----------+----------+--------------------+--------------------+-----------+-----------+-----------+-----------+-----------+--------------------+-------------------+------------+--------------------+-------------+-------------+-------------+-------------+--------------+--------------+--------------------+--------------+---------------+--------------------+---------------+--------------------+---------------+---------------+---------------+--------------------+----------------+----------------+--------------------+----------------+----------------+-----------------+--------------------+-------------------+--------------------+--------------------+--------------------+-----------

In [4]:
USEFUL_COLS = ["Id", "Code", "Description", "IdProductItem"]
df_selected = df_filtered.select(USEFUL_COLS)
df_selected.show(5)

+-----+----------+--------------------+-------------+
|   Id|      Code|         Description|IdProductItem|
+-----+----------+--------------------+-------------+
|11840|    125851|   ACCOR ATF (Accor)|          186|
|11841|  75W80/2L|ACCOR SYNTHO (Accor)|          186|
|11842|  75W90/2L|ACCOR SYNCHO 75W9...|          186|
|11843|EFM0000404|   Pompe à carburant|          187|
|11844|EFM0000407|        POMPE +CORPS|          187|
+-----+----------+--------------------+-------------+
only showing top 5 rows



In [5]:
df_clean = df_selected.withColumn(
    "IdProductItem",
    F.coalesce(F.col("IdProductItem"), F.lit(0))
)

# Vérifier qu'il n'y a plus de NULL
null_count = df_clean.filter(F.col("IdProductItem").isNull()).count()
print("NULL restants dans IdProductItem :", null_count)
df_clean.show(5)

NULL restants dans IdProductItem : 0
+-----+----------+--------------------+-------------+
|   Id|      Code|         Description|IdProductItem|
+-----+----------+--------------------+-------------+
|11840|    125851|   ACCOR ATF (Accor)|          186|
|11841|  75W80/2L|ACCOR SYNTHO (Accor)|          186|
|11842|  75W90/2L|ACCOR SYNCHO 75W9...|          186|
|11843|EFM0000404|   Pompe à carburant|          187|
|11844|EFM0000407|        POMPE +CORPS|          187|
+-----+----------+--------------------+-------------+
only showing top 5 rows



In [7]:
# Ajoute cette cellule AVANT la cellule GE pour inspecter l'API
import great_expectations as gx
context = gx.get_context(mode="ephemeral")
help(context.suites.add)

Help on method add in module great_expectations.core.factory.suite_factory:

add(suite: 'ExpectationSuite') -> 'ExpectationSuite' method of great_expectations.core.factory.suite_factory.SuiteFactory instance
    --Public API--Add an ExpectationSuite to the collection.
    
    Parameters:
        suite: ExpectationSuite to add
    
    Raises:
        DataContextError if ExpectationSuite already exists



In [6]:
from utils.ge_runner import run_ge_validation
from expectations.item_expectations import get_item_suite

run_ge_validation(df_clean, "item_suite", get_item_suite, "item")

Calculating Metrics:   0%|          | 0/23 [00:00<?, ?it/s]

✅ Validation GX OK pour item


{
  "success": true,
  "results": [
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_not_be_null",
        "kwargs": {
          "batch_id": "item-item",
          "column": "Id"
        },
        "meta": {},
        "id": "c9a451bd-64a2-41c3-9039-3890504d7c17"
      },
      "result": {
        "element_count": 65678,
        "unexpected_count": 0,
        "unexpected_percent": 0.0,
        "partial_unexpected_list": [],
        "partial_unexpected_counts": [],
        "partial_unexpected_index_list": []
      },
      "meta": {},
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message": null
      }
    },
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_be_unique",
        "kwargs": {
          "batch_id": "item-item",
          "column": "Id"
        },
        "meta": {},
        "id": "35bfc835-2048-405d-b17d-4

In [7]:
from utils.silver_writer import write_silver

write_silver(df_clean, "silver.item")
print("Vérification lecture silver.item :")
spark.read.format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/warehouse_db") \
    .option("dbtable", '"silver"."item"') \
    .option("user", "warehouse") \
    .option("password", "warehouse") \
    .option("driver", "org.postgresql.Driver") \
    .load().show(5)

[Stage 13:>                                                         (0 + 1) / 1]

✅ Écriture terminée → silver.item
Vérification lecture silver.item :
+-----+----------+--------------------+-------------+
|   Id|      Code|         Description|IdProductItem|
+-----+----------+--------------------+-------------+
|11840|    125851|   ACCOR ATF (Accor)|          186|
|11841|  75W80/2L|ACCOR SYNTHO (Accor)|          186|
|11842|  75W90/2L|ACCOR SYNCHO 75W9...|          186|
|11843|EFM0000404|   Pompe à carburant|          187|
|11844|EFM0000407|        POMPE +CORPS|          187|
+-----+----------+--------------------+-------------+
only showing top 5 rows



In [8]:
# Cellule diagnostic — voir la table ProductItem
df_product = spark.read.format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/warehouse_db") \
    .option("dbtable", '"bronze"."ProductItem"') \
    .option("user", "warehouse") \
    .option("password", "warehouse") \
    .option("driver", "org.postgresql.Driver") \
    .load()

df_product.printSchema()
df_product.show(5)

root
 |-- _airbyte_raw_id: string (nullable = true)
 |-- _airbyte_extracted_at: timestamp (nullable = true)
 |-- _airbyte_meta: string (nullable = true)
 |-- _airbyte_generation_id: long (nullable = true)
 |-- Id: long (nullable = true)
 |-- IdTecDoc: long (nullable = true)
 |-- IsDeleted: boolean (nullable = true)
 |-- UrlPicture: string (nullable = true)
 |-- CodeProduct: string (nullable = true)
 |-- UpdatedDate: timestamp (nullable = true)
 |-- CreationDate: timestamp (nullable = true)
 |-- LabelProduct: string (nullable = true)
 |-- Deleted_Token: string (nullable = true)
 |-- TransactionUserEmail: string (nullable = true)



[Stage 15:>                                                         (0 + 1) / 1]

+--------------------+---------------------+--------------------+----------------------+---+--------+---------+--------------------+-----------+--------------------+------------+------------+--------------------+--------------------+
|     _airbyte_raw_id|_airbyte_extracted_at|       _airbyte_meta|_airbyte_generation_id| Id|IdTecDoc|IsDeleted|          UrlPicture|CodeProduct|         UpdatedDate|CreationDate|LabelProduct|       Deleted_Token|TransactionUserEmail|
+--------------------+---------------------+--------------------+----------------------+---+--------+---------+--------------------+-----------+--------------------+------------+------------+--------------------+--------------------+
|019d3b25-f643-7d6...| 2026-03-29 19:50:...|{"changes": [], "...|                     1|185|    NULL|     true|                NULL|        ABR| 2024-01-27 22:45:05|        NULL|         ABR|a377fd8b-19bd-45e...|                NULL|
|019d3b25-f6a7-70e...| 2026-03-29 19:50:...|{"changes": [], "...

In [9]:
# Dans le notebook
regex = ".*[a-zA-Z0-9].*"
df_pandas = df_clean.toPandas()

mask = (
    df_pandas["Code"].notna() & 
    ~df_pandas["Code"].astype(str).str.contains(regex, regex=True, na=False)
)

print(df_pandas.loc[mask, ["Id", "Code", "Description", "IdProductItem"]])

          Id                   Code                        Description  \
143    12026                      -                        PLATEAU EMB   
144    12027                      +                         C/B CB2016   
1153   13049                     **                                 **   
1199   13094                    *--                                 --   
1293   13193                 ******                              **/**   
4616   16773             **********                           ********   
4618   16778                     **                                 **   
4622   16790                   ****                                ***   
5364   17587                   ....                    KIT EMB GII 1.5   
5424   17647                      +                              .....   
5574   18009                  .....                               ....   
5800   18036                  *****                                ***   
6974   19279                      -   

In [10]:
df_clean.toPandas()["Code"].dropna().sample(20).tolist()

['CO9602',
 '102 641',
 '530 0538 10',
 'TH6251.92J',
 'AD0254904',
 '500 1199 10',
 '1 457 433 593',
 '04C260849E',
 '6RF199262J',
 'SR6352',
 '3AA810972',
 '03687',
 'BKCD0815',
 '11291001601',
 '20739',
 '668032',
 '1K0807244A',
 'VG0562123',
 '105-02-202',
 'N  90959101']

In [11]:
df_raw = spark.read.format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/warehouse_db") \
    .option("dbtable", '"bronze"."ProductItem"') \
    .option("user", "warehouse") \
    .option("password", "warehouse") \
    .option("driver", "org.postgresql.Driver") \
    .load()

print("Nombre de lignes :", df_raw.count())
df_raw.printSchema()
df_raw.show(5)

Nombre de lignes : 940
root
 |-- _airbyte_raw_id: string (nullable = true)
 |-- _airbyte_extracted_at: timestamp (nullable = true)
 |-- _airbyte_meta: string (nullable = true)
 |-- _airbyte_generation_id: long (nullable = true)
 |-- Id: long (nullable = true)
 |-- IdTecDoc: long (nullable = true)
 |-- IsDeleted: boolean (nullable = true)
 |-- UrlPicture: string (nullable = true)
 |-- CodeProduct: string (nullable = true)
 |-- UpdatedDate: timestamp (nullable = true)
 |-- CreationDate: timestamp (nullable = true)
 |-- LabelProduct: string (nullable = true)
 |-- Deleted_Token: string (nullable = true)
 |-- TransactionUserEmail: string (nullable = true)

+--------------------+---------------------+--------------------+----------------------+---+--------+---------+--------------------+-----------+--------------------+------------+------------+--------------------+--------------------+
|     _airbyte_raw_id|_airbyte_extracted_at|       _airbyte_meta|_airbyte_generation_id| Id|IdTecDoc|IsDel

In [13]:
# Identifier la ligne échouée dans LabelProduct
df_pandas = df_raw.toPandas()

# NULL
nulls = df_pandas.loc[df_pandas["LabelProduct"].isna(), ["Id", "LabelProduct"]]
print("NULLs :")
print(nulls)

# Bizarres
mask_bizarre = (
    df_pandas["LabelProduct"].notna() & 
    ~df_pandas["LabelProduct"].str.match(r"^[a-zA-Z0-9].*")
)
print("\nValeurs bizarres :")
print(df_pandas.loc[mask_bizarre, ["Id", "LabelProduct"]])

NULLs :
Empty DataFrame
Columns: [Id, LabelProduct]
Index: []

Valeurs bizarres :
       Id LabelProduct
303  4127    ÜRO Parts


In [2]:
df_raw = spark.read.format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/warehouse_db") \
    .option("dbtable", '"bronze"."Document"') \
    .option("user", "warehouse") \
    .option("password", "warehouse") \
    .option("driver", "org.postgresql.Driver") \
    .load()

In [3]:
# Lance ça d'abord pour voir les types et quelques valeurs
df_raw.select(["Id", "Code", "IdDocumentStatus", "DocumentTypeCode", 
               "IdTiers", "DocumentDate", "DocumentTTCPrice", "IsForPos"]).printSchema()

df_raw.select(["Id", "Code", "IdDocumentStatus", "DocumentTypeCode", 
               "IdTiers", "DocumentDate", "DocumentTTCPrice", "IsForPos"]).show(5)

root
 |-- Id: long (nullable = true)
 |-- Code: string (nullable = true)
 |-- IdDocumentStatus: long (nullable = true)
 |-- DocumentTypeCode: string (nullable = true)
 |-- IdTiers: long (nullable = true)
 |-- DocumentDate: timestamp (nullable = true)
 |-- DocumentTTCPrice: decimal(38,18) (nullable = true)
 |-- IsForPos: boolean (nullable = true)



+------+-------------+----------------+----------------+-------+--------------------+--------------------+--------+
|    Id|         Code|IdDocumentStatus|DocumentTypeCode|IdTiers|        DocumentDate|    DocumentTTCPrice|IsForPos|
+------+-------------+----------------+----------------+-------+--------------------+--------------------+--------+
|560195| BL2500026450|               3|            D-SA|   1901|2025-04-07 16:31:...|8.025000000000000000|   false|
|560196| BL2500026451|               3|            D-SA|   1586|2025-04-07 16:32:...|109.4290000000000...|   false|
|560197| L25-00007203|               3|            D-PU|   3494|2025-04-07 20:32:...|197.7270000000000...|    NULL|
|560198| L25-00007204|               3|            D-PU|   3494|2025-04-07 19:33:...|295.9850000000000...|    NULL|
|560199|FA25-00003238|               6|            I-SA|  12200|2025-04-07 17:33:...|374.9990000000000...|   false|
+------+-------------+----------------+----------------+-------+--------

In [4]:
# Voir la répartition IsForPos par DocumentTypeCode
df_raw.groupBy("DocumentTypeCode", "IsForPos").count().orderBy("DocumentTypeCode").show()

[Stage 1:>                                                          (0 + 1) / 1]

+----------------+--------+------+
|DocumentTypeCode|IsForPos| count|
+----------------+--------+------+
|            A-PU|    NULL|   761|
|            A-PU|   false|   365|
|            A-SA|    true|  9095|
|            A-SA|   false|  7972|
|            A-SA|    NULL|    42|
|            B-PU|   false|    19|
|           BE-PU|    NULL|  1958|
|           BE-PU|   false|   952|
|           BS-SA|    NULL|  1332|
|           BS-SA|   false|   712|
|            D-PU|    NULL| 63949|
|            D-PU|   false| 32581|
|            D-SA|    true|230632|
|            D-SA|   false|239783|
|           FO-PU|    NULL|   223|
|           FO-PU|   false|  7665|
|            I-PU|   false| 13877|
|            I-PU|    NULL|  3499|
|            I-SA|    true|  6257|
|            I-SA|   false| 47536|
+----------------+--------+------+
only showing top 20 rows



In [10]:
df_lines = spark.read.format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/warehouse_db") \
    .option("dbtable", '"bronze"."DocumentLine"') \
    .option("user", "warehouse") \
    .option("password", "warehouse") \
    .option("driver", "org.postgresql.Driver") \
    .load()

In [16]:
from pyspark.sql import functions as F

df_joined = df_lines.join(
    df_raw,
    df_lines["IdItem"] == df_raw["Id"],
    how="left"
)

# Maintenant groupBy sur IdNature
df_joined.groupBy("IdNature") \
    .agg(
        F.min("MovementQty").alias("Min"),
        F.max("MovementQty").alias("Max"),
        F.avg("MovementQty").alias("Moyenne")
    ) \
    .orderBy("IdNature") \
    .show()

[Stage 8:>                                                          (0 + 1) / 1]

+--------+--------------------+--------------------+--------------------+
|IdNature|                 Min|                 Max|             Moyenne|
+--------+--------------------+--------------------+--------------------+
|       1|1.000000000000000000|11800.00000000000...|7.385284841710247...|
|       2|0.000000000000000000|30453.00000000000...|3.778174946083627...|
|       3|1.000000000000000000|1.000000000000000000|1.000000000000000...|
|       4|1.000000000000000000|1.000000000000000000|1.000000000000000...|
|       5|1.000000000000000000|1.000000000000000000|1.000000000000000...|
|       6|-1.00000000000000...|-1.00000000000000...|-1.00000000000000...|
+--------+--------------------+--------------------+--------------------+



In [18]:
# Cell 1 — Spark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("test_item") \
    .config("spark.jars", "/opt/spark/jars/postgresql.jar") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

In [19]:
# Cell 2 — Extract
JDBC_URL  = "jdbc:postgresql://postgres:5432/warehouse_db"
JDBC_OPTS = {"user": "warehouse", "password": "warehouse", "driver": "org.postgresql.Driver"}

df_raw = spark.read.format("jdbc") \
    .option("url", JDBC_URL) \
    .option("dbtable", '"bronze"."Item"') \
    .options(**JDBC_OPTS) \
    .load()

In [20]:
# Cell 3 — Transform
import sys
sys.path.insert(0, "/opt/jobs")
from clean_item import transform, extract_product_item

df_product = extract_product_item(spark)
df_clean = transform(df_raw, df_product)
print(f"Lignes : {df_clean.count()}")

Lignes : 65678


In [23]:
import great_expectations as gx

context = gx.get_context(mode="ephemeral")

# Créer la suite d'abord
from great_expectations.core.expectation_suite import ExpectationSuite
suite = ExpectationSuite(name="item_profiled")
suite = context.suites.add(suite)

# Créer datasource
datasource = context.data_sources.add_pandas(name="profiling_item")
data_asset = datasource.add_dataframe_asset(name="item")
batch_def  = data_asset.add_batch_definition_whole_dataframe("item_batch")

df_pandas = df_clean.toPandas()
batch = batch_def.get_batch(batch_parameters={"dataframe": df_pandas})

# Valider et afficher résultats
validation_result = batch.validate(suite)

# Stats basiques pandas
print("=== Shape ===")
print(df_pandas.shape)

print("\n=== Types ===")
print(df_pandas.dtypes)

print("\n=== NULLs ===")
print(df_pandas.isnull().sum())

print("\n=== Valeurs uniques ===")
for col in df_pandas.columns:
    print(f"{col} : {df_pandas[col].nunique()} valeurs uniques")

Calculating Metrics: 0it [00:00, ?it/s]

=== Shape ===
(65678, 5)

=== Types ===
Id                int64
Code             object
Label            object
IdProductItem    object
IdFamily         object
dtype: object

=== NULLs ===
Id               0
Code             0
Label            0
IdProductItem    0
IdFamily         0
dtype: int64

=== Valeurs uniques ===
Id : 65678 valeurs uniques
Code : 63819 valeurs uniques
Label : 49739 valeurs uniques
IdProductItem : 595 valeurs uniques
IdFamily : 95 valeurs uniques
